<a href="https://colab.research.google.com/github/Mati1491/Tp_final/blob/main/tp_programacion_avanzada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from datetime import date, timedelta

LIMITE_PRESTAMOS = 3
DIAS_PRESTAMO = 14
MULTA_POR_DIA = 50

class Biblioteca:
    def __init__(self):
        self.usuarios = []
        self.libros = []
        self.prestamos = []

    def alta_usuario(self, usuario):
        # Revisamos primero si el usuario ingresado ya existe con el DNI
        for i in self.usuarios:
            if i.dni == usuario.dni:
                print(f"El usuario {i.dni} ya está ingresado")
                return

        # Si el usuario no está agregado, lo agregamos a la lista
        self.usuarios.append(usuario)
        print(f"Usuario {usuario.dni} registrado")

    def baja_usuario(self, dni):
        """ Lo que hace el enumerate es convertir la lista en pares donde i es
        el índice de la lista y u el usuario en esa posición """
        for i, u in enumerate(self.usuarios):
            if u.dni == dni:
                del self.usuarios[i] # Elimina el usuario de la lista
                print(f"Usuario con DNI {dni} eliminado")
                return

        # Esta línea se ejecuta SOLO si el for terminó sin encontrar el DNI
        print(f"No se encontró usuario con DNI {dni}")

    def alta_libro(self, libro):
        for i in self.libros:
            if i.isbn == libro.isbn:
                print(f"El libro {libro.isbn} ya está registrado")
                return

        # Si no está en la lista de libros, lo agrega
        self.libros.append(libro)
        print(f"Libro {libro.isbn} ha sido registrado")

    def baja_libro(self, isbn):
        """ Lo que hace el enumerate es convertir la lista en pares donde i es
        el índice de la lista y l el libro en esa posición """
        for i, l in enumerate(self.libros):
            if l.isbn == isbn:
                del self.libros[i] # Elimina el libro de la lista
                print(f"Libro con ISBN {isbn} eliminado")
                return

        # Esta línea se ejecuta solo si el for terminó sin encontrar el ISBN
        print(f"No se encontró el libro con ISBN {isbn}")

    def modificar_libro(self, isbn, nuevo_titulo=None, nuevo_autor=None, nuevo_anio=None, nuevas_paginas=None):
        # Busco el libro en la lista usando su ISBN
        for libro in self.libros:
            if libro.isbn == isbn:
                # Si el usuario pasó un dato nuevo (no es None), lo cambio en el libro
                if nuevo_titulo is not None:
                    libro.titulo = nuevo_titulo
                if nuevo_autor is not None:
                    libro.autor = nuevo_autor
                if nuevo_anio is not None:
                    libro.anio_publicacion = nuevo_anio
                if nuevas_paginas is not None:
                    libro.cantidad_paginas = nuevas_paginas

                # Aviso que todo salió bien y corto el método con return
                print("Libro modificado con éxito")
                return

        # Si el bucle terminó y no entró al "if", aviso que el libro no existe
        print("No se encontró un libro con ese ISBN")

    def modificar_usuario(self, dni, nuevo_nombre=None, nuevo_apellido=None, nuevo_email=None):
        # Recorremos  la lista de usuarios para buscarlo por su DNI
        for usuario in self.usuarios:
            if usuario.dni == dni:
                # Reviso qué datos me mandaron para actualizar y los cambio
                if nuevo_nombre is not None:
                    usuario.nombre = nuevo_nombre
                if nuevo_apellido is not None:
                    usuario.apellido = nuevo_apellido
                if nuevo_email is not None:
                    usuario.email = nuevo_email

                # Mostramos que se cambio el usuio
                print("Usuario modificado con éxito")
                return

        # Si el DNI no coincidió con ninguno, avisamos que
        print("No se encontró un Usuario con ese DNI")

    def buscar_usuario(self, dni):
        for usuario in self.usuarios:
            if usuario.dni == dni:
                return usuario
        return None

    def buscar_libro(self, isbn):
        for libro in self.libros:
            if libro.isbn == isbn:
                return libro
        return None

    def _prestamos_activos_usuario(self, dni):
        return sum(1 for p in self.prestamos
                   if p.usuario.dni == dni and not p.devuelto)

    def registrar_prestamo(self, dni, isbn, dias=DIAS_PRESTAMO):
        usuario = self.buscar_usuario(dni)
        libro   = self.buscar_libro(isbn)

        if usuario is None:
            print("Usuario no encontrado")
            return
        if libro is None:
            print("Libro no encontrado")
            return
        if self._prestamos_activos_usuario(dni) >= LIMITE_PRESTAMOS:
            print(f"{usuario.nombre_completo()} ya tiene {LIMITE_PRESTAMOS} préstamos activos")
            return

        for prestamo in self.prestamos:
            if prestamo.libro.isbn == isbn and not prestamo.devuelto:
                print("El libro ya tiene un préstamo activo")
                return

        nuevo_prestamo = Prestamo(usuario, libro, dias=dias)
        self.prestamos.append(nuevo_prestamo)
        print(f"Préstamo registrado — vence el {nuevo_prestamo.fecha_limite}")

    def devolver_libro(self, isbn):
        for prestamo in self.prestamos:
            if prestamo.libro.isbn == isbn and not prestamo.devuelto:
                multa = prestamo.calcular_multa()
                prestamo.registrar_devolucion()
                if multa > 0:
                    print(f"Libro devuelto con atraso — Multa: ${multa}")
                else:
                    print("Libro devuelto correctamente")
                return
        print("No existe préstamo activo para ese libro")

    def listar_prestamos_activos(self):
        print("\n=== PRÉSTAMOS ACTIVOS ===")
        hay_prestamos = False
        for prestamo in self.prestamos:
            if not prestamo.devuelto:
                print(prestamo)
                hay_prestamos = True
        if not hay_prestamos:
            print("No hay préstamos activos")


class Libro:
    def __init__(self, titulo, autor, isbn, anio_publicacion, cantidad_paginas):
        self.titulo = titulo
        self.autor = autor
        self.isbn = isbn
        self.anio_publicacion = anio_publicacion
        self.cantidad_paginas = cantidad_paginas


class Persona:
    def __init__(self, nombre, apellido, dni):
        self.nombre = nombre
        self.apellido = apellido
        self.dni = dni


class Usuario(Persona):
    def __init__(self, nombre, apellido, dni, email):
        super().__init__(nombre, apellido, dni)
        self.email = email

    def nombre_completo(self):
        return f"{self.nombre} {self.apellido}"


class Prestamo:
    def __init__(self, usuario, libro, fecha_prestamo=None, dias=DIAS_PRESTAMO):
        self.usuario= usuario
        self.libro = libro
        self.fecha_prestamo = fecha_prestamo or date.today()
        self.fecha_limite = self.fecha_prestamo + timedelta(days=dias)
        self.fecha_devolucion = None
        self.devuelto = False

    def esta_vencido(self):
        return not self.devuelto and date.today() > self.fecha_limite

    def calcular_multa(self):
        if not self.esta_vencido():
            return 0
        dias_atraso = (date.today() - self.fecha_limite).days
        return dias_atraso * MULTA_POR_DIA

    def registrar_devolucion(self, fecha=None):
        self.fecha_devolucion = fecha or date.today()
        self.devuelto = True

    def __str__(self):
        if self.devuelto:
            return (f"Préstamo de '{self.libro.titulo}' a {self.usuario.nombre_completo()} "
                    f"| Devuelto el {self.fecha_devolucion}")
        estado = "⚠ VENCIDO" if self.esta_vencido() else "Prestamo activo"
        multa  = f" — Multa: ${self.calcular_multa()}" if self.esta_vencido() else ""
        return (f"Préstamo de '{self.libro.titulo}' a {self.usuario.nombre_completo()} "
                f"| Vence: {self.fecha_limite} | {estado}{multa}")


In [13]:
#PRUEBAS
mi_biblioteca = Biblioteca()

# Creamos usuarios y libros
user1 = Usuario("Juan", "Pérez", "12345678", "juan@email.com")
libro1 = Libro("El Aleph", "Jorge Luis Borges", "978-123", 1949, 150)
libro2 = Libro("Ficciones", "Jorge Luis Borges", "978-456", 1944, 200)
libro3 = Libro("Rayuela", "Julio Cortázar", "978-789", 1963, 600)
libro4 = Libro("El túnel", "Ernesto Sabato", "978-000", 1948, 140)

print("--- Probando Altas ---")
mi_biblioteca.alta_usuario(user1)
mi_biblioteca.alta_libro(libro1)
mi_biblioteca.alta_libro(libro2)
mi_biblioteca.alta_libro(libro3)
mi_biblioteca.alta_libro(libro4)

print("\n--- Probando Modificaciones ---")
mi_biblioteca.modificar_usuario("12345678", nuevo_apellido="Gómez")
mi_biblioteca.modificar_libro("978-123", nuevo_titulo="El Aleph (Edición Especial)")

print("\n--- Probando límite de préstamos ---")
mi_biblioteca.registrar_prestamo("12345678", "978-123")
mi_biblioteca.registrar_prestamo("12345678", "978-456")
mi_biblioteca.registrar_prestamo("12345678", "978-789")
mi_biblioteca.registrar_prestamo("12345678", "978-000")  # este debe fallar

mi_biblioteca.listar_prestamos_activos()

print("\n--- Probando Devolución ---")
mi_biblioteca.devolver_libro("978-123")
mi_biblioteca.listar_prestamos_activos()

print("\n--- Probando multa ---")
prestamo_vencido = Prestamo(user1, libro4,
                            fecha_prestamo=date.today() - timedelta(days=20),
                            dias=14)
mi_biblioteca.prestamos.append(prestamo_vencido)
mi_biblioteca.listar_prestamos_activos()  # tiene que aparecer VENCIDO con multa

--- Probando Altas ---
Usuario 12345678 registrado
Libro 978-123 ha sido registrado
Libro 978-456 ha sido registrado
Libro 978-789 ha sido registrado
Libro 978-000 ha sido registrado

--- Probando Modificaciones ---
Usuario modificado con éxito
Libro modificado con éxito

--- Probando límite de préstamos ---
Préstamo registrado — vence el 2026-07-12
Préstamo registrado — vence el 2026-07-12
Préstamo registrado — vence el 2026-07-12
Juan Gómez ya tiene 3 préstamos activos

=== PRÉSTAMOS ACTIVOS ===
Préstamo de 'El Aleph (Edición Especial)' a Juan Gómez | Vence: 2026-07-12 | Prestamo activo
Préstamo de 'Ficciones' a Juan Gómez | Vence: 2026-07-12 | Prestamo activo
Préstamo de 'Rayuela' a Juan Gómez | Vence: 2026-07-12 | Prestamo activo

--- Probando Devolución ---
Libro devuelto correctamente

=== PRÉSTAMOS ACTIVOS ===
Préstamo de 'Ficciones' a Juan Gómez | Vence: 2026-07-12 | Prestamo activo
Préstamo de 'Rayuela' a Juan Gómez | Vence: 2026-07-12 | Prestamo activo

--- Probando multa ---